# 提示
为 MCP 客户端创建可重复使用的参数化提示模板。

提示符是可重复使用的消息模板，可帮助 LLM 生成结构化、有目的性的响应。FastMCP 主要使用@mcp.prompt装饰器来简化这些模板的定义。

​
## 什么是提示？
提示为 LLM 提供参数化的消息模板。当客户端请求提示时：

- FastMCP 找到相应的提示定义。
- 如果它有参数，则会根据您的函数签名进行验证。
- 您的函数将使用经过验证的输入执行。
- 生成的消息将返回给 LLM 以指导其响应。

这使得您可以定义 LLM 可以在不同的客户端和环境中使用的一致、可重复使用的模板。

## 提示
​
`@prompt装饰`
定义提示符最常见的方式是装饰一个 Python 函数。装饰器使用函数名作为提示符的标识符。

In [ ]:
from fastmcp import FastMCP
from fastmcp.prompts.prompt import Message, PromptMessage, TextContent

mcp = FastMCP(name="PromptServer")

# Basic prompt returning a string (converted to user message automatically)
@mcp.prompt()
def ask_about_topic(topic: str) -> str:
    """Generates a user message asking for an explanation of a topic."""
    return f"Can you please explain the concept of '{topic}'?"

# Prompt returning a specific message type
@mcp.prompt()
def generate_code_request(language: str, task_description: str) -> PromptMessage:
    """Generates a user message requesting code generation."""
    content = f"Write a {language} function that performs the following task: {task_description}"
    return PromptMessage(role="user", content=TextContent(type="text", text=content))

关键概念：

- 名称：默认情况下，提示名称取自函数名称。
- 参数：函数参数定义生成提示所需的输入。
- 推断元数据：默认情况下：
- 提示名称：取自函数名称（ask_about_topic）。
- 提示描述：取自函数的文档字符串。

## 返回值
FastMCP 智能地处理提示函数的不同返回类型：

- `str`：自动转换为单个`PromptMessage`。
- `PromptMessage`：直接使用提供的参数。（请注意，我们提供了一个更友好的`Message`构造函数，它可以接受原始字符串而不是TextContent对象。）
- `list[PromptMessage | str]`：用作一系列消息（对话）。
- `Any`：如果返回类型不属于上述类型，则会尝试将返回值转换为字符串，并用作 `PromptMessage`



In [ ]:
from fastmcp.prompts.prompt import Message

@mcp.prompt()
def roleplay_scenario(character: str, situation: str) -> list[Message]:
    """Sets up a roleplaying scenario with initial messages."""
    return [
        Message(f"Let's roleplay. You are {character}. The situation is: {situation}"),
        Message("Okay, I understand. I am ready. What happens next?", role="assistant")
    ]

## 类型注解
类型注释对于提示非常重要。它们：

- 告知 FastMCP 每个参数的预期类型。
- 允许验证从客户端收到的参数。
- 用于生成 MCP 协议的提示模式。

In [ ]:
from pydantic import Field
from typing import Literal, Optional

@mcp.prompt()
def generate_content_request(
    topic: str = Field(description="The main subject to cover"),
    format: Literal["blog", "email", "social"] = "blog",
    tone: str = "professional",
    word_count: Optional[int] = None
) -> str:
    """Create a request for generating content in a specific format."""
    prompt = f"Please write a {format} post about {topic} in a {tone} tone."
    
    if word_count:
        prompt += f" It should be approximately {word_count} words long."
        
    return prompt

### 必需参数与可选参数
除非函数签名中的参数具有默认值，否则它们将被视为必需的。

In [ ]:
@mcp.prompt()
def data_analysis_prompt(
    data_uri: str,                        # Required - no default value
    analysis_type: str = "summary",       # Optional - has default value
    include_charts: bool = False          # Optional - has default value
) -> str:
    """Creates a request to analyze data with specific parameters."""
    prompt = f"Please perform a '{analysis_type}' analysis on the data found at {data_uri}."
    if include_charts:
        prompt += " Include relevant charts and visualizations."
    return prompt

在本例中，客户端必须提供 `data_uri`。如果省略 `analysis_type` 或 `include_charts`，将使用它们的默认值。

## 提示元数据
虽然 FastMCP 从您的函数中推断出名称和描述，但您可以覆盖这些并使用`@mcp.prompt`装饰器的参数添加标签：




In [ ]:
@mcp.prompt(
    name="analyze_data_request",          # Custom prompt name
    description="Creates a request to analyze data with specific parameters",  # Custom description
    tags={"analysis", "data"}             # Optional categorization tags
)
def data_analysis_prompt(
    data_uri: str = Field(description="The URI of the resource containing the data."),
    analysis_type: str = Field(default="summary", description="Type of analysis.")
) -> str:
    """This docstring is ignored when description is provided."""
    return f"Please perform a '{analysis_type}' analysis on the data found at {data_uri}."

- name：设置通过 MCP 公开的明确提示名称。
- description：提供通过 MCP 公开的描述。如果设置，则函数的文档字符串将被忽略。
- tags：用于对提示进行分类的一组字符串。客户端可以使用标签来筛选或分组可用的提示。
​


## 异步提示
FastMCP 无缝支持标准（def）和异步（async def）函数作为提示。

async def当提示函数执行 I/O 操作（如网络请求、数据库查询、文件 I/O 或外部服务调用）时使用。

In [ ]:
# Synchronous prompt
@mcp.prompt()
def simple_question(question: str) -> str:
    """Generates a simple question to ask the LLM."""
    return f"Question: {question}"

# Asynchronous prompt
@mcp.prompt()
async def data_based_prompt(data_id: str) -> str:
    """Generates a prompt based on data that needs to be fetched."""
    # In a real scenario, you might fetch data from a database or API
    async with aiohttp.ClientSession() as session:
        async with session.get(f"https://api.example.com/data/{data_id}") as response:
            data = await response.json()
            return f"Analyze this data: {data['content']}"

## 访问 MCP 上下文

提示可以通过该对象访问额外的 MCP 信息和功能`Context`。要访问它，请向提示函数添加一个参数，并在其类型注释中注明`Context`：

In [ ]:
from fastmcp import FastMCP, Context

mcp = FastMCP(name="PromptServer")

@mcp.prompt()
async def generate_report_request(report_type: str, ctx: Context) -> str:
    """Generates a request for a report."""
    return f"Please create a {report_type} report. Request ID: {ctx.request_id}"

## 服务器行为
### 重复提示
您可以配置 FastMCP 服务器如何处理尝试注册多个同名提示的情况。请FastMCP在初始化期间使用此`on_duplicate_prompts`设置。


In [ ]:
from fastmcp import FastMCP

mcp = FastMCP(
    name="PromptServer",
    on_duplicate_prompts="error"  # Raise an error if a prompt name is duplicated
)

@mcp.prompt()
def greeting(): return "Hello, how can I help you today?"

# This registration attempt will raise a ValueError because
# "greeting" is already registered and the behavior is "error".
# @mcp.prompt()
# def greeting(): return "Hi there! What can I do for you?"

- "warn"（默认）：记录警告，新提示将替换旧提示。
- "error"：提出ValueError，防止重复注册。
- "replace"：默默地用新提示替换现有提示。
- "ignore"：保留原始提示并忽略新的注册尝试。